# 11a — Cross-Validation Pipeline & Feature Selection

This notebook implements feature selection and 5-fold stratified cross-validation using the integrated **demographic + questionnaire** dataset produced by the multimodal data integration stage.

## Objective

The notebook:

- loads `data/processed/demographics_questionnaire.csv` as the modeling dataset;
- confirms the participant-level structure of the available integrated datasets;
- uses `StratifiedGroupKFold` with `patient_id` as the grouping variable to align with the grouped-validation strategy;
- excludes participant identifiers and diagnosis-derived variables from predictors;
- fits preprocessing and feature selection **inside each training fold**;
- applies the fitted transformations and selected feature set to the corresponding validation fold;
- evaluates fold-level performance using a class-weighted multinomial Logistic Regression model;
- exports selected feature sets and selected training/validation feature datasets for each fold;
- produces feature-selection stability and fold-performance summaries;
- verifies that validation information is not used when fitting preprocessing or feature selection.

The selected features produced in this notebook are used primarily for **stability analysis**. Feature selection should be re-fit within the later hyperparameter-tuning pipeline rather than reusing one fixed feature set from these folds.

## 1. Libraries and project paths

In [ ]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import SelectKBest, VarianceThreshold, f_classif
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 150)

# Locate the project root whether the notebook is executed from
# the repository root or from the notebooks directory.
cwd = Path.cwd().resolve()

if (cwd / "src").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "src").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError(
        "Could not locate the project root containing the src directory."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
METRICS_DIR = OUTPUTS_DIR / "metrics"
TABLES_DIR = OUTPUTS_DIR / "tables"
FIGURES_DIR = OUTPUTS_DIR / "figures"

for directory in [METRICS_DIR, TABLES_DIR, FIGURES_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

DATA_FILE = PROCESSED_DIR / "demographics_questionnaire.csv"

# Additional integrated datasets are checked only to confirm
# participant-level structure. They are NOT used as modeling
# inputs in this notebook.
INTEGRATED_DATA_FILES = {
    "demographics_questionnaire": PROCESSED_DIR / "demographics_questionnaire.csv",
    "wearable_questionnaire": PROCESSED_DIR / "wearable_questionnaire.csv",
    "multimodal_full": PROCESSED_DIR / "multimodal_full.csv",
}

print(f"Project root: {PROJECT_ROOT}")
print(f"Input dataset: {DATA_FILE}")
print(f"Metrics directory: {METRICS_DIR}")
print(f"Tables directory: {TABLES_DIR}")
print(f"Figures directory: {FIGURES_DIR}")

## 2. Cross-validation and feature-selection configuration

In [ ]:
RANDOM_STATE = 42
N_SPLITS = 5
N_SELECTED_FEATURES = 20
TARGET_COLUMN = "label"
PARTICIPANT_ID_COLUMN = "patient_id"

# Columns excluded from modeling because they are identifiers
# or diagnosis-derived variables that could cause leakage.
EXCLUDED_COLUMNS = [
    PARTICIPANT_ID_COLUMN,
    TARGET_COLUMN,
    "condition_group",
]

print(f"Cross-validation folds: {N_SPLITS}")
print(f"Requested selected features per fold: {N_SELECTED_FEATURES}")
print(f"Random state: {RANDOM_STATE}")

`N_SELECTED_FEATURES = 20` is used as a fixed starting point so that feature-selection stability can be compared consistently across all five folds. The value is **not treated as an optimal final choice**. The number of selected features should be included as a tunable hyperparameter during the later model-tuning stage.

Selection is performed after fold-specific preprocessing using an ANOVA F-test (`SelectKBest(f_classif)`). A zero-variance filter is applied first so features with no variability in a training fold cannot be selected.

## 3. Load the demographic + questionnaire dataset

In [ ]:
assert DATA_FILE.exists(), (
    f"Dataset not found: {DATA_FILE}\n"
    "Run the multimodal data integration notebook first."
)

data = pd.read_csv(
    DATA_FILE,
    dtype={PARTICIPANT_ID_COLUMN: str},
)

print(f"Dataset shape: {data.shape}")
print(
    f"Unique participants: "
    f"{data[PARTICIPANT_ID_COLUMN].nunique():,}"
)
display(data.head())

The integrated dataset contains one participant-level record per row and combines demographic variables with questionnaire-derived features. The target is `label`; `patient_id` is retained only for split auditing and output traceability.

## 4. Confirm participant-level structure across integrated datasets

The professor suggested grouped validation as an option. To determine whether grouping is necessary, the available integrated datasets are checked for repeated `patient_id` values. These additional files are used **only for this structural check**; the modeling pipeline below continues to use only `demographics_questionnaire.csv`.

In [ ]:
dataset_structure_records = []

for dataset_name, dataset_path in INTEGRATED_DATA_FILES.items():
    if dataset_path.exists():
        check_df = pd.read_csv(
            dataset_path,
            dtype={PARTICIPANT_ID_COLUMN: str},
        )

        dataset_structure_records.append({
            "dataset": dataset_name,
            "rows": len(check_df),
            "unique_participants": check_df[PARTICIPANT_ID_COLUMN].nunique(),
            "duplicate_patient_rows": (
                check_df[PARTICIPANT_ID_COLUMN]
                .duplicated()
                .sum()
            ),
            "max_rows_per_participant": (
                check_df[PARTICIPANT_ID_COLUMN]
                .value_counts()
                .max()
            ),
        })
    else:
        dataset_structure_records.append({
            "dataset": dataset_name,
            "rows": np.nan,
            "unique_participants": np.nan,
            "duplicate_patient_rows": np.nan,
            "max_rows_per_participant": np.nan,
        })

dataset_structure_summary = pd.DataFrame(
    dataset_structure_records
)

display(dataset_structure_summary)

available_structure = dataset_structure_summary.dropna(
    subset=["rows"]
)

assert (
    available_structure["max_rows_per_participant"]
    .eq(1)
    .all()
), (
    "At least one integrated dataset contains repeated participant rows. "
    "If repeated rows are present, use StratifiedGroupKFold with patient_id."
)

print(
    "Participant-level structure check: PASS — "
    "all available integrated datasets contain one row per participant."
)
print(
    "StratifiedGroupKFold is used to explicitly preserve participant-level "
    "grouping while maintaining class balance across folds."
)

Although each participant currently appears only once, `StratifiedGroupKFold` is used to explicitly enforce participant-level grouping with `patient_id` while also preserving class balance across folds. This keeps the validation strategy consistent with the agreed grouped-validation approach and remains appropriate if repeated participant observations are introduced later.

## 5. Dataset validation

In [ ]:
required_columns = {
    PARTICIPANT_ID_COLUMN,
    TARGET_COLUMN,
}

missing_required = required_columns.difference(data.columns)

assert not missing_required, (
    f"Missing required columns: {sorted(missing_required)}"
)

assert data[PARTICIPANT_ID_COLUMN].notna().all(), (
    "Missing participant identifiers detected."
)

duplicate_participants = data[
    data[PARTICIPANT_ID_COLUMN].duplicated(keep=False)
]

assert duplicate_participants.empty, (
    "Duplicate participant rows detected. Because this dataset should "
    "contain one row per participant, duplicates must be resolved before "
    "using StratifiedKFold."
)

assert data[TARGET_COLUMN].notna().all(), (
    "Missing target labels detected."
)

print("Dataset validation: PASS")
print(f"Rows: {len(data):,}")
print(
    f"Unique participants: "
    f"{data[PARTICIPANT_ID_COLUMN].nunique():,}"
)

print("\nClass distribution:")
display(
    data[TARGET_COLUMN]
    .value_counts()
    .sort_index()
    .rename("count")
    .to_frame()
)

## 6. Define predictors and target

In [ ]:
feature_columns = [
    column
    for column in data.columns
    if column not in EXCLUDED_COLUMNS
]

X = data[feature_columns].copy()
y = data[TARGET_COLUMN].copy()
participant_ids = data[PARTICIPANT_ID_COLUMN].copy()

print(f"Raw predictor count: {X.shape[1]}")
print(f"Target classes: {sorted(y.unique().tolist())}")

print("\nExcluded columns:")
print(EXCLUDED_COLUMNS)

print("\nPredictor columns:")
print(feature_columns)

`condition_group` is excluded because it is diagnosis-derived and therefore directly related to the target. `patient_id` is excluded from model fitting but retained separately so participant overlap can be checked for every fold.

## 7. Identify numerical and categorical predictors

In [ ]:
numerical_features = (
    X.select_dtypes(include=["number"])
    .columns
    .tolist()
)

categorical_features = (
    X.select_dtypes(exclude=["number"])
    .columns
    .tolist()
)

feature_type_summary = pd.DataFrame({
    "feature_type": ["numerical", "categorical"],
    "count": [
        len(numerical_features),
        len(categorical_features),
    ],
})

display(feature_type_summary)

print("\nNumerical features:")
print(numerical_features)

print("\nCategorical features:")
print(categorical_features)

## 8. Build fold-specific preprocessing

In [ ]:
def build_preprocessor(
    numerical_columns,
    categorical_columns,
):
    """Create a fresh preprocessing transformer for one CV fold."""

    numerical_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(strategy="median"),
            ),
            (
                "scaler",
                StandardScaler(),
            ),
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(strategy="most_frequent"),
            ),
            (
                "encoder",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False,
                ),
            ),
        ]
    )

    return ColumnTransformer(
        transformers=[
            (
                "numerical",
                numerical_pipeline,
                numerical_columns,
            ),
            (
                "categorical",
                categorical_pipeline,
                categorical_columns,
            ),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )

A new preprocessor is created and fitted for each cross-validation fold. Numerical imputation medians, categorical imputation modes, scaling parameters, and one-hot encoding categories are therefore learned exclusively from the training portion of that fold.

## 9. Initialize stratified cross-validation

In [ ]:
cv = StratifiedGroupKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

print(cv)

`StratifiedGroupKFold` is used with `patient_id` as the grouping variable to align with the grouped-validation strategy discussed for the project. The demographic + questionnaire dataset currently contains one row per participant, so grouping does not materially change the split compared with standard stratification; however, it explicitly guarantees that all records associated with the same participant remain within a single fold while maintaining class balance as closely as possible.

## 10. Run leakage-safe cross-validation and feature selection

In [ ]:
fold_metrics = []
selected_feature_records = []
fold_audit_records = []
fold_assignment_records = []
out_of_fold_predictions = []

for fold_number, (train_index, validation_index) in enumerate(
    cv.split(X, y, groups=participant_ids),
    start=1,
):
    # ----------------------------------------------------------
    # Partition raw participant-level data
    # ----------------------------------------------------------
    X_train = X.iloc[train_index].copy()
    X_validation = X.iloc[validation_index].copy()

    y_train = y.iloc[train_index].copy()
    y_validation = y.iloc[validation_index].copy()

    train_ids = participant_ids.iloc[train_index].copy()
    validation_ids = participant_ids.iloc[validation_index].copy()

    # Explicit participant-level leakage check.
    participant_overlap = (
        set(train_ids)
        .intersection(set(validation_ids))
    )

    assert not participant_overlap, (
        f"Participant overlap detected in fold {fold_number}: "
        f"{sorted(participant_overlap)}"
    )

    # ----------------------------------------------------------
    # Fresh preprocessing + selection + classifier for this fold
    # ----------------------------------------------------------
    preprocessor = build_preprocessor(
        numerical_features,
        categorical_features,
    )

    pipeline = Pipeline(
        steps=[
            (
                "preprocessing",
                preprocessor,
            ),
            (
                "variance_filter",
                VarianceThreshold(threshold=0.0),
            ),
            (
                "feature_selection",
                SelectKBest(
                    score_func=f_classif,
                    k=N_SELECTED_FEATURES,
                ),
            ),
            (
                "classifier",
                LogisticRegression(
                    solver="lbfgs",
                    max_iter=1000,
                    class_weight="balanced",
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    )

    # IMPORTANT:
    # fit() receives TRAINING-FOLD data only.
    # The validation fold is untouched until prediction/transform.
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        pipeline.fit(
            X_train,
            y_train,
        )

    validation_predictions = pipeline.predict(
        X_validation
    )

    # ----------------------------------------------------------
    # Fold-level performance
    # ----------------------------------------------------------
    metrics = {
        "fold": fold_number,
        "train_participants": len(train_index),
        "validation_participants": len(validation_index),
        "accuracy": accuracy_score(
            y_validation,
            validation_predictions,
        ),
        "balanced_accuracy": balanced_accuracy_score(
            y_validation,
            validation_predictions,
        ),
        "macro_f1": f1_score(
            y_validation,
            validation_predictions,
            average="macro",
            zero_division=0,
        ),
        "precision_macro": precision_score(
            y_validation,
            validation_predictions,
            average="macro",
            zero_division=0,
        ),
        "recall_macro": recall_score(
            y_validation,
            validation_predictions,
            average="macro",
            zero_division=0,
        ),
    }

    fold_metrics.append(metrics)

    # ----------------------------------------------------------
    # Recover selected transformed feature names
    # ----------------------------------------------------------
    preprocessing_step = pipeline.named_steps[
        "preprocessing"
    ]

    transformed_feature_names = (
        preprocessing_step
        .get_feature_names_out()
    )

    variance_step = pipeline.named_steps[
        "variance_filter"
    ]

    variance_feature_names = (
        transformed_feature_names[
            variance_step.get_support()
        ]
    )

    selection_step = pipeline.named_steps[
        "feature_selection"
    ]

    selected_feature_names = (
        variance_feature_names[
            selection_step.get_support()
        ]
    )

    selected_scores = (
        selection_step.scores_[
            selection_step.get_support()
        ]
    )

    assert len(selected_feature_names) == N_SELECTED_FEATURES

    # ----------------------------------------------------------
    # Record selected features
    # ----------------------------------------------------------
    fold_selected_table = pd.DataFrame({
        "fold": fold_number,
        "feature": selected_feature_names,
        "selection_score": selected_scores,
    }).sort_values(
        "selection_score",
        ascending=False,
    )

    fold_selected_table.to_csv(
        TABLES_DIR
        / f"cv_fold_{fold_number}_selected_features.csv",
        index=False,
    )

    selected_feature_records.extend(
        fold_selected_table.to_dict("records")
    )

    # ----------------------------------------------------------
    # Export selected train/validation feature datasets
    # using ONLY the fold-fitted preprocessing + selector.
    # ----------------------------------------------------------
    X_train_preprocessed = (
        preprocessing_step.transform(X_train)
    )

    X_validation_preprocessed = (
        preprocessing_step.transform(X_validation)
    )

    X_train_variance = variance_step.transform(
        X_train_preprocessed
    )

    X_validation_variance = variance_step.transform(
        X_validation_preprocessed
    )

    X_train_selected = selection_step.transform(
        X_train_variance
    )

    X_validation_selected = selection_step.transform(
        X_validation_variance
    )

    train_selected_df = pd.DataFrame(
        X_train_selected,
        columns=selected_feature_names,
    )

    train_selected_df.insert(
        0,
        TARGET_COLUMN,
        y_train.reset_index(drop=True),
    )

    train_selected_df.insert(
        0,
        PARTICIPANT_ID_COLUMN,
        train_ids.reset_index(drop=True),
    )

    validation_selected_df = pd.DataFrame(
        X_validation_selected,
        columns=selected_feature_names,
    )

    validation_selected_df.insert(
        0,
        TARGET_COLUMN,
        y_validation.reset_index(drop=True),
    )

    validation_selected_df.insert(
        0,
        PARTICIPANT_ID_COLUMN,
        validation_ids.reset_index(drop=True),
    )

    train_selected_df.to_csv(
        TABLES_DIR
        / f"cv_fold_{fold_number}_selected_train_dataset.csv",
        index=False,
    )

    validation_selected_df.to_csv(
        TABLES_DIR
        / f"cv_fold_{fold_number}_selected_validation_dataset.csv",
        index=False,
    )

    # ----------------------------------------------------------
    # Leakage audit
    # ----------------------------------------------------------
    fold_audit_records.append({
        "fold": fold_number,
        "train_participants": train_ids.nunique(),
        "validation_participants": validation_ids.nunique(),
        "participant_overlap_count": len(participant_overlap),
        "preprocessing_fit_scope": "training_fold_only",
        "feature_selection_fit_scope": "training_fold_only",
        "selected_feature_count": len(selected_feature_names),
        "leakage_check": "PASS",
    })

    # Save fold membership for reproducibility.
    for participant_id in train_ids:
        fold_assignment_records.append({
            PARTICIPANT_ID_COLUMN: participant_id,
            "fold": fold_number,
            "role": "train",
        })

    for participant_id in validation_ids:
        fold_assignment_records.append({
            PARTICIPANT_ID_COLUMN: participant_id,
            "fold": fold_number,
            "role": "validation",
        })

    # Out-of-fold predictions for optional aggregate checks.
    out_of_fold_predictions.extend(
        pd.DataFrame({
            PARTICIPANT_ID_COLUMN:
                validation_ids.reset_index(drop=True),
            "fold": fold_number,
            "y_true":
                y_validation.reset_index(drop=True),
            "y_pred":
                validation_predictions,
        }).to_dict("records")
    )

    print(
        f"Fold {fold_number}: "
        f"train={len(train_index)}, "
        f"validation={len(validation_index)}, "
        f"selected={len(selected_feature_names)}, "
        f"macro_f1={metrics['macro_f1']:.3f}"
    )

The validation subset is not supplied to `.fit()` at any stage. Preprocessing statistics, encoded categories, zero-variance filtering, ANOVA feature scores, selected feature indices, and classifier coefficients are all estimated from the corresponding training fold only. The fitted transformations are then applied to that fold's validation data.

## 11. Fold-level performance summary

In [ ]:
fold_performance = pd.DataFrame(
    fold_metrics
)

display(fold_performance)

fold_performance.to_csv(
    METRICS_DIR
    / "cv_fold_performance.csv",
    index=False,
)

In [ ]:
metric_columns = [
    "accuracy",
    "balanced_accuracy",
    "macro_f1",
    "precision_macro",
    "recall_macro",
]

cv_performance_summary = pd.DataFrame({
    "metric": metric_columns,
    "mean": [
        fold_performance[column].mean()
        for column in metric_columns
    ],
    "std": [
        fold_performance[column].std(ddof=1)
        for column in metric_columns
    ],
    "min": [
        fold_performance[column].min()
        for column in metric_columns
    ],
    "max": [
        fold_performance[column].max()
        for column in metric_columns
    ],
})

display(cv_performance_summary)

cv_performance_summary.to_csv(
    METRICS_DIR
    / "cv_performance_summary.csv",
    index=False,
)

## 12. Selected features by fold

In [ ]:
selected_features_by_fold = pd.DataFrame(
    selected_feature_records
)

display(
    selected_features_by_fold
    .sort_values(
        ["fold", "selection_score"],
        ascending=[True, False],
    )
)

selected_features_by_fold.to_csv(
    TABLES_DIR
    / "cv_selected_features_all_folds.csv",
    index=False,
)

## 13. Feature-selection stability summary

In [ ]:
feature_selection_summary = (
    selected_features_by_fold
    .groupby("feature")
    .agg(
        times_selected=("fold", "nunique"),
        mean_selection_score=("selection_score", "mean"),
        std_selection_score=("selection_score", "std"),
    )
    .reset_index()
)

feature_selection_summary[
    "selection_percentage"
] = (
    feature_selection_summary[
        "times_selected"
    ]
    / N_SPLITS
    * 100
)

feature_selection_summary = (
    feature_selection_summary
    .sort_values(
        [
            "times_selected",
            "mean_selection_score",
        ],
        ascending=[False, False],
    )
    .reset_index(drop=True)
)

display(feature_selection_summary)

feature_selection_summary.to_csv(
    TABLES_DIR
    / "cv_feature_selection_summary.csv",
    index=False,
)

Features selected repeatedly across folds are more stable under resampling. The `times_selected` and `selection_percentage` fields quantify selection stability, while the mean selection score summarizes the average univariate discrimination observed when the feature was selected.

The fold-specific selected features are used here to assess **selection stability**, not as a fixed feature list for the next tuning notebook. During hyperparameter tuning, the feature-selection step should remain inside the tuning pipeline and be fitted again within the training folds.

## 14. Leakage verification and fold audit

In [ ]:
fold_audit = pd.DataFrame(
    fold_audit_records
)

display(fold_audit)

assert (
    fold_audit["participant_overlap_count"]
    .eq(0)
    .all()
)

assert (
    fold_audit["leakage_check"]
    .eq("PASS")
    .all()
)

assert (
    fold_audit["feature_selection_fit_scope"]
    .eq("training_fold_only")
    .all()
)

fold_audit.to_csv(
    TABLES_DIR
    / "cv_feature_selection_leakage_audit.csv",
    index=False,
)

print(
    "Leakage verification: PASS — "
    "preprocessing and feature selection were fit "
    "using training-fold data only."
)

In [ ]:
fold_assignments = pd.DataFrame(
    fold_assignment_records
)

fold_assignments.to_csv(
    TABLES_DIR
    / "cv_fold_assignments.csv",
    index=False,
)

# Every participant must serve as validation exactly once.
validation_counts = (
    fold_assignments[
        fold_assignments["role"] == "validation"
    ]
    [PARTICIPANT_ID_COLUMN]
    .value_counts()
)

assert validation_counts.eq(1).all()
assert len(validation_counts) == len(data)

print(
    "Fold assignment verification: PASS — "
    "every participant appears in validation exactly once."
)

## 15. Out-of-fold prediction summary

In [ ]:
oof_predictions = pd.DataFrame(
    out_of_fold_predictions
)

assert (
    oof_predictions[PARTICIPANT_ID_COLUMN]
    .nunique()
    == len(data)
)

oof_metrics = pd.DataFrame(
    [{
        "accuracy": accuracy_score(
            oof_predictions["y_true"],
            oof_predictions["y_pred"],
        ),
        "balanced_accuracy":
            balanced_accuracy_score(
                oof_predictions["y_true"],
                oof_predictions["y_pred"],
            ),
        "macro_f1": f1_score(
            oof_predictions["y_true"],
            oof_predictions["y_pred"],
            average="macro",
            zero_division=0,
        ),
        "precision_macro": precision_score(
            oof_predictions["y_true"],
            oof_predictions["y_pred"],
            average="macro",
            zero_division=0,
        ),
        "recall_macro": recall_score(
            oof_predictions["y_true"],
            oof_predictions["y_pred"],
            average="macro",
            zero_division=0,
        ),
    }]
)

display(oof_metrics)

oof_predictions.to_csv(
    TABLES_DIR
    / "cv_out_of_fold_predictions.csv",
    index=False,
)

oof_metrics.to_csv(
    METRICS_DIR
    / "cv_out_of_fold_metrics.csv",
    index=False,
)

## 16. Feature-selection stability figure

In [ ]:
stable_features = (
    feature_selection_summary
    .head(20)
    .sort_values(
        [
            "times_selected",
            "mean_selection_score",
        ],
        ascending=[True, True],
    )
)

fig, ax = plt.subplots(
    figsize=(9, 7)
)

ax.barh(
    stable_features["feature"],
    stable_features["selection_percentage"],
)

ax.set_xlabel(
    "Selection frequency across folds (%)"
)

ax.set_ylabel("Feature")

ax.set_title(
    "Top Feature Selection Stability Across 5 CV Folds"
)

ax.set_xlim(0, 100)

fig.tight_layout()

figure_path = (
    FIGURES_DIR
    / "cv_feature_selection_stability.png"
)

fig.savefig(
    figure_path,
    dpi=300,
    bbox_inches="tight",
)

plt.show()

print(f"Saved: {figure_path}")

## 17. Fold performance figure

In [ ]:
fig, ax = plt.subplots(
    figsize=(8, 5)
)

ax.plot(
    fold_performance["fold"],
    fold_performance["macro_f1"],
    marker="o",
)

ax.set_xlabel("Cross-validation fold")
ax.set_ylabel("Macro F1")
ax.set_title(
    "Macro F1 Across 5 Stratified CV Folds"
)
ax.set_xticks(
    fold_performance["fold"]
)

fig.tight_layout()

figure_path = (
    FIGURES_DIR
    / "cv_fold_macro_f1.png"
)

fig.savefig(
    figure_path,
    dpi=300,
    bbox_inches="tight",
)

plt.show()

print(f"Saved: {figure_path}")

## 18. Aggregate confusion matrix

In [ ]:
class_labels = sorted(
    data[TARGET_COLUMN].unique().tolist()
)

aggregate_confusion = confusion_matrix(
    oof_predictions["y_true"],
    oof_predictions["y_pred"],
    labels=class_labels,
)

aggregate_confusion_df = pd.DataFrame(
    aggregate_confusion,
    index=[
        f"True_{label}"
        for label in class_labels
    ],
    columns=[
        f"Pred_{label}"
        for label in class_labels
    ],
)

display(aggregate_confusion_df)

aggregate_confusion_df.to_csv(
    METRICS_DIR
    / "cv_out_of_fold_confusion_matrix.csv",
)

fig, ax = plt.subplots(
    figsize=(6, 5)
)

image = ax.imshow(
    aggregate_confusion
)

fig.colorbar(image, ax=ax)

ax.set_title(
    "Out-of-Fold Confusion Matrix"
)
ax.set_xlabel("Predicted label")
ax.set_ylabel("True label")

ax.set_xticks(
    range(len(class_labels))
)
ax.set_yticks(
    range(len(class_labels))
)

ax.set_xticklabels(class_labels)
ax.set_yticklabels(class_labels)

for i in range(
    aggregate_confusion.shape[0]
):
    for j in range(
        aggregate_confusion.shape[1]
    ):
        ax.text(
            j,
            i,
            aggregate_confusion[i, j],
            ha="center",
            va="center",
        )

fig.tight_layout()

figure_path = (
    FIGURES_DIR
    / "cv_out_of_fold_confusion_matrix.png"
)

fig.savefig(
    figure_path,
    dpi=300,
    bbox_inches="tight",
)

plt.show()

print(f"Saved: {figure_path}")

## 19. Deliverables verification

In [ ]:
deliverables = pd.DataFrame({
    "deliverable": [
        "5-fold stratified cross-validation pipeline",
        "Feature selection within each training fold",
        "Selected feature list for each fold",
        "Selected training feature dataset for each fold",
        "Selected validation feature dataset for each fold",
        "Feature selection summary",
        "Fold performance summary",
        "Leakage verification audit",
        "Out-of-fold predictions and metrics",
        "Feature-selection stability figure",
        "Fold macro-F1 figure",
        "Out-of-fold confusion matrix",
    ],
    "location": [
        "Notebook",
        "Notebook",
        "outputs/tables/cv_fold_*_selected_features.csv",
        "outputs/tables/cv_fold_*_selected_train_dataset.csv",
        "outputs/tables/cv_fold_*_selected_validation_dataset.csv",
        "outputs/tables/cv_feature_selection_summary.csv",
        "outputs/metrics/cv_fold_performance.csv and cv_performance_summary.csv",
        "outputs/tables/cv_feature_selection_leakage_audit.csv",
        "outputs/tables/cv_out_of_fold_predictions.csv and outputs/metrics/cv_out_of_fold_metrics.csv",
        "outputs/figures/cv_feature_selection_stability.png",
        "outputs/figures/cv_fold_macro_f1.png",
        "outputs/metrics/cv_out_of_fold_confusion_matrix.csv and outputs/figures/cv_out_of_fold_confusion_matrix.png",
    ],
    "status": ["READY"] * 12,
})

deliverables

## 20. Initial observations

In [ ]:
best_fold = (
    fold_performance
    .sort_values(
        "macro_f1",
        ascending=False,
    )
    .iloc[0]
)

most_stable = (
    feature_selection_summary[
        feature_selection_summary[
            "times_selected"
        ]
        == N_SPLITS
    ]
    ["feature"]
    .tolist()
)

print("CROSS-VALIDATION SUMMARY")
print("-" * 50)

print(
    "Mean macro F1: "
    f"{fold_performance['macro_f1'].mean():.3f} "
    "± "
    f"{fold_performance['macro_f1'].std(ddof=1):.3f}"
)

print(
    "Mean balanced accuracy: "
    f"{fold_performance['balanced_accuracy'].mean():.3f} "
    "± "
    f"{fold_performance['balanced_accuracy'].std(ddof=1):.3f}"
)

print(
    f"Best fold by macro F1: "
    f"Fold {int(best_fold['fold'])} "
    f"({best_fold['macro_f1']:.3f})"
)

print(
    "Features selected in all five folds: "
    f"{len(most_stable)}"
)

if most_stable:
    print(most_stable)

print(
    "\nInterpret fold-to-fold variability together with "
    "feature-selection stability before treating any individual "
    "feature as consistently predictive."
)

## 21. Conclusion

This notebook implements a reproducible five-fold **StratifiedGroupKFold** cross-validation workflow using the demographic + questionnaire participant-level dataset. `patient_id` is supplied as the grouping variable so that participant-level separation is explicitly enforced while class proportions are preserved as closely as possible across folds. Although the current integrated dataset contains one row per participant, this grouped strategy aligns with the agreed validation approach and remains robust if repeated participant observations are introduced later.

Every fold constructs and fits a new preprocessing pipeline, zero-variance filter, ANOVA feature selector, and class-weighted Logistic Regression classifier using training-fold data only. `N_SELECTED_FEATURES = 20` is treated as an initial fixed value for stability analysis and should be optimized during later hyperparameter tuning.

The selected features generated here are intended primarily to evaluate feature-selection stability across folds. They should not be reused as one fixed feature set during tuning; instead, feature selection should be re-fit within the tuning pipeline. The exported fold-specific datasets, feature-selection summaries, fold-performance metrics, and audit outputs provide the required artifacts for downstream model development and technical reporting.